# Phase 3 — DistilBERT Fine-Tuning
**ClearSignal — Person 2**

### Setup (do this once before running)
1. Runtime → Change runtime type → **T4 GPU**
2. Upload just **3 files** using the cell below (or the Files panel on the left):
   - `train_features.csv`
   - `val_features.csv`
   - `test_features.csv`

### What you get back (download after running)
- `bert_weights/` folder → save to your local `models_saved/bert_weights/`
- `bert_val_preds.npy` → save to your local `models_saved/bert_val_preds.npy`
- `bert_test_preds.npy` → save to your local `models_saved/bert_test_preds.npy`

**Expected finding:** BERT will likely underperform Ridge — synthetic transcripts give it no real language signal. Document this in the report.

In [ ]:
# Cell 1: Install dependencies
!pip install transformers torch pandas numpy scikit-learn -q

In [ ]:
# Cell 2: Upload your 3 CSV files
# This opens a file picker — select train_features.csv, val_features.csv, test_features.csv
from google.colab import files
uploaded = files.upload()  # upload all 3 at once
print('Uploaded:', list(uploaded.keys()))

In [ ]:
# Cell 3: Config — no paths to change
import os
import torch
import numpy as np
import pandas as pd

TRAIN_CSV = 'train_features.csv'
VAL_CSV   = 'val_features.csv'
TEST_CSV  = 'test_features.csv'

BERT_WEIGHTS_DIR   = 'bert_weights'
VAL_PREDS_PATH     = 'bert_val_preds.npy'
TEST_PREDS_PATH    = 'bert_test_preds.npy'

LR         = 2e-5
BATCH_SIZE = 8
MAX_EPOCHS = 3
MAX_LENGTH = 512

os.makedirs(BERT_WEIGHTS_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cpu':
    print('[WARN] No GPU — change runtime to T4 GPU for reasonable speed')

In [ ]:
# Cell 4: Load data
def clean_df(df):
    # Drop duplicate .1 columns
    mask = ~df.columns.duplicated(keep='first')
    df = df.loc[:, mask].copy()
    df.drop(columns=[c for c in df.columns if c.endswith('.1')], inplace=True, errors='ignore')
    return df

train_df = clean_df(pd.read_csv(TRAIN_CSV)).dropna(subset=['transcript_text', 'csat_score'])
val_df   = clean_df(pd.read_csv(VAL_CSV)).dropna(subset=['transcript_text', 'csat_score'])
test_df  = clean_df(pd.read_csv(TEST_CSV)).dropna(subset=['transcript_text', 'csat_score'])

print(f'Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}')
print(f'CSAT range: [{train_df.csat_score.min()}, {train_df.csat_score.max()}]')

In [ ]:
# Cell 5: Dataset + tokenizer
from transformers import DistilBertTokenizer
from torch.utils.data import Dataset, DataLoader

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

class CSATDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(
            list(texts), truncation=True, max_length=MAX_LENGTH,
            padding=True, return_tensors='pt'
        )
        self.labels = torch.tensor(list(labels), dtype=torch.float32)
    def __len__(self): return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

train_ds = CSATDataset(train_df['transcript_text'], train_df['csat_score'])
val_ds   = CSATDataset(val_df['transcript_text'],   val_df['csat_score'])
test_ds  = CSATDataset(test_df['transcript_text'],  test_df['csat_score'])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

# Verify truncation works
sample = tokenizer(['x ' * 5000], truncation=True, max_length=MAX_LENGTH, return_tensors='pt')
assert sample['input_ids'].shape[1] <= MAX_LENGTH
print(f'Tokenizer OK — max_length={MAX_LENGTH} enforced')

In [ ]:
# Cell 6: Build model
from transformers import DistilBertForSequenceClassification

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased', num_labels=1, problem_type='regression'
).to(device)

print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Cell 7: Fine-tune with early stopping
from torch.optim import AdamW
from sklearn.metrics import mean_absolute_error

optimizer = AdamW(model.parameters(), lr=LR)
best_val_loss = float('inf')
no_improve = 0
best_val_preds = None

for epoch in range(1, MAX_EPOCHS + 1):
    # Train
    model.train()
    train_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        out = model(
            input_ids=batch['input_ids'].to(device),
            attention_mask=batch['attention_mask'].to(device),
            labels=batch['labels'].to(device)
        )
        out.loss.backward()
        optimizer.step()
        train_loss += out.loss.item()
    train_loss /= len(train_loader)

    # Validate
    model.eval()
    val_loss, preds, labels = 0.0, [], []
    with torch.no_grad():
        for batch in val_loader:
            out = model(
                input_ids=batch['input_ids'].to(device),
                attention_mask=batch['attention_mask'].to(device),
                labels=batch['labels'].to(device)
            )
            val_loss += out.loss.item()
            p = out.logits.squeeze(-1).cpu().numpy()
            preds.extend(p.tolist() if p.ndim > 0 else [float(p)])
            labels.extend(batch['labels'].numpy().tolist())
    val_loss /= len(val_loader)
    val_preds_clipped = np.clip(preds, 1.0, 5.0)
    val_mae = mean_absolute_error(labels, val_preds_clipped)

    print(f'Epoch {epoch}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}  val_MAE={val_mae:.4f}')

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        no_improve = 0
        model.save_pretrained(BERT_WEIGHTS_DIR)
        tokenizer.save_pretrained(BERT_WEIGHTS_DIR)
        best_val_preds = val_preds_clipped.copy()
        print(f'  → Saved best weights')
    else:
        no_improve += 1
        if no_improve >= 1:
            print(f'  Early stopping at epoch {epoch}')
            break

np.save(VAL_PREDS_PATH, best_val_preds)
print(f'\nSaved val preds → {VAL_PREDS_PATH}')
print(f'BERT val MAE: {mean_absolute_error(labels, best_val_preds):.4f}')

In [ ]:
# Cell 8: Generate test set predictions (needed by evaluate.py)
# Load best weights
best_model = DistilBertForSequenceClassification.from_pretrained(BERT_WEIGHTS_DIR).to(device)
best_model.eval()

test_preds = []
with torch.no_grad():
    for batch in test_loader:
        out = best_model(
            input_ids=batch['input_ids'].to(device),
            attention_mask=batch['attention_mask'].to(device)
        )
        p = out.logits.squeeze(-1).cpu().numpy()
        test_preds.extend(p.tolist() if p.ndim > 0 else [float(p)])

test_preds_clipped = np.clip(test_preds, 1.0, 5.0)
np.save(TEST_PREDS_PATH, test_preds_clipped)

test_mae = mean_absolute_error(test_df['csat_score'].values, test_preds_clipped)
print(f'Saved test preds → {TEST_PREDS_PATH}')
print(f'BERT test MAE: {test_mae:.4f}')

# FINDING check
ridge_val_mae = 1.1295  # from your Phase 1 output
bert_val_mae = mean_absolute_error(labels, best_val_preds)
if bert_val_mae > ridge_val_mae:
    print(f'\n[FINDING] BERT val MAE ({bert_val_mae:.4f}) > Ridge val MAE ({ridge_val_mae:.4f})')
    print('→ Expected. Synthetic transcripts give BERT no real language signal.')
    print('  Document this in the final report.')

In [ ]:
# Cell 9: Download everything you need
import shutil
from google.colab import files

# Download val and test predictions
files.download(VAL_PREDS_PATH)
files.download(TEST_PREDS_PATH)

# Zip and download the weights folder
shutil.make_archive('bert_weights', 'zip', '.', BERT_WEIGHTS_DIR)
files.download('bert_weights.zip')

print('Downloaded:')
print(f'  bert_val_preds.npy  → put in models_saved/')
print(f'  bert_test_preds.npy → put in models_saved/')
print(f'  bert_weights.zip    → unzip into models_saved/bert_weights/')
print()
print('Then locally:')
print('  Set BERT_READY = True in src/models/ensemble.py and src/evaluation/evaluate.py')
print('  python src/models/ensemble.py')
print('  python src/evaluation/evaluate.py')